# Orbit width

`deposit` and `slow_down` keep every fast ion on the flux shell of its birth point: zero orbit width. A fast ion moves on a guiding-centre orbit that leaves that surface, by the conservation of the canonical toroidal momentum $p_\varphi = m R v_\parallel b_\varphi + e\psi$: passing ions by a drift-surface shift of about $q\rho_\parallel$, trapped ones over a banana of width about $q\rho_L/\sqrt{\epsilon}$. Codes that slow the ions down locally correct the deposition for it by averaging each birth over its first orbit (NUBEAM's first-orbit redistribution, Pankin et al., Comput. Phys. Commun. 159, 157, 2004; RABBIT's orbit-averaged births, Weiland et al., Nucl. Fusion 58, 082032, 2018).

`orbit_average_births(deposition, equilibrium)` (in `freeofcharge.orbit_width`) does this: it follows each birth's collisionless guiding-centre orbit (the compiled pusher of `freeofcharge.orbits`) for one period, a poloidal transit or a bounce, splits the birth's weight over the shells by the time the orbit spends in each, and removes the births whose first orbit leaves the plasma (prompt losses). The result is a `Deposition`, so `slow_down` takes it unchanged: first orbit, then local slowing down.

In [ ]:
import time
from pathlib import Path

import numpy as np
import plasma_plots as pp
import xarray as xr
from plasmacoll.constants import ATOMIC_MASS
from scipy.constants import mu_0

import freeofcharge as foc
from freeofcharge import ascot5, orbits
from freeofcharge.orbit_width import orbit_average_births

pp.set_backend("plotly")
pp.set_math("latex")


def models(arrays, names, name, units):
    """Stack profiles over a 'model' dimension for plasma-plots."""
    out = xr.concat(arrays, dim=xr.DataArray(names, dims="model", name="model"))
    out.name = name
    out.attrs.update(long_name=name, units=units)
    return out

## One birth, one orbit

The tokamak of the [ASCOT5 benchmark](/freeofcharge/benchmarks/ascot5/) (`ascot5.benchmark_case`: a Solov'ev equilibrium of ASDEX Upgrade's size, 2.5 T, $q = 1$ on the axis, a wall on $\psi_n = 1.3$) with its 60 keV deuterium beam, 200 markers. A 60 keV deuteron has a Larmor radius of 2 cm here, and banana widths of several cm against a minor radius of 50 cm. The period of an orbit ends where its guiding centre returns through a line across its starting point in the poloidal plane, which closes passing orbits, bananas and the potatoes near the axis alike. Prompt losses are counted at the wall ($\rho = \sqrt{1.3}$, `rho_limit`); the time an orbit spends outside $\rho = 1$ carries no weight.

Three births and their first orbits, with the flux surfaces $\rho$ = 0.25, 0.5, 0.75 and 1:

In [ ]:
case = ascot5.benchmark_case(markers=200)
equilibrium, plasma = case.equilibrium, case.plasma
rho_edges = case.rho_edges  # 20 shells
surfaces = equilibrium.flux_surfaces(rho_edges)
field = orbits.orbit_field(equilibrium, (128, 4, 128))
wall_rho = np.sqrt(1.3)
deposition = foc.deposit(
    case.beam,
    equilibrium,
    plasma,
    step=0.005,
    births_per_marker=1,
    rho_edges=rho_edges,
    seed=2,
)
start = time.perf_counter()
first = orbit_average_births(deposition, equilibrium, field=field, rho_limit=wall_rho)
print(f"{len(first.orbit_time)} births averaged in {time.perf_counter() - start:.1f} s")
print(
    f"trapped {first.trapped.mean():.0%}, prompt losses {first.prompt_loss_fraction:.2%} of the power, "
    f"orbit periods {1e6 * np.median(first.orbit_time):.0f} us (median)"
)

# a passing birth, a trapped one and the one that moves most
confined = ~first.prompt_lost
birth_rho = np.asarray(deposition.birth_rho)
shift = np.abs(first.mean_rho - birth_rho)
picks = [
    np.nonzero(confined & ~first.trapped)[0][0],
    np.nonzero(confined & first.trapped & (birth_rho > 0.3))[0][0],
    int(np.nanargmax(np.where(confined, shift, np.nan))),
]
births = deposition.births
markers = orbits.guiding_centres(
    np.asarray(births.positions)[picks],
    np.asarray(births.velocities)[picks],
    np.ones(3),
    field,
    deposition.mass_number * ATOMIC_MASS,
)
with pp.figure(figsize=(5, 6)) as fig:
    ax = fig[0]
    for value in (0.25, 0.5, 0.75, 1.0):
        surface = equilibrium.flux_surface(value**2, 256)
        ax.plot(surface[:, 0], surface[:, 1], color="0.7", linewidth=1)
    for k, j in enumerate(picks):
        one = orbits.GuidingCentres(
            markers.positions[k : k + 1],
            markers.v_parallel[k : k + 1],
            markers.mu[k : k + 1],
            np.ones(1),
            markers.mass,
        )
        run = orbits.trace_orbits(
            one, field, first.orbit_time[j], samples=400, step=0.01
        )
        kind = "trapped" if first.trapped[j] else "passing"
        ax.plot(
            run.R.values[:, 0],
            run.z.values[:, 0],
            linewidth=1.5,
            label=f"{kind}, born at rho {birth_rho[j]:.2f}",
        )
        shells = np.nonzero(first.shell_fractions[j] > 0.01)[0]
        print(
            f"{kind:8s} born at rho {birth_rho[j]:.2f}, pitch {float(deposition.birth_pitch[j]):+.2f}: "
            f"over rho {rho_edges[shells[0]]:.2f} to {rho_edges[shells[-1] + 1]:.2f}, mean {first.mean_rho[j]:.2f}"
        )
    ax.set_aspect("equal")
    ax.set_xlabel("$R$ [m]")
    ax.set_ylabel("$Z$ [m]")
    ax.set_title("First orbits")
    ax.legend()

The passing ion (pitch 0.76, born at $\rho = 0.67$) stays within $\rho$ = 0.60 to 0.70, the trapped one born at 0.77 spreads over 0.75 to 0.85, and the ion that moves most, trapped and born at $\rho = 0.24$, spans 0.20 to 0.45 with its time-weighted mean at 0.34: near the axis a banana covers several shells of width 0.05. Half of these births are trapped and none is lost at the wall. An orbit period is about 16 µs, so the 200 births take a fraction of a second.

## Deposition and heating in an ITER-like plasma

The ITER-like case of [Heating and current drive](/freeofcharge/examples/slowing-down-and-current-drive/): a Solov'ev equilibrium ($R_0 = 6.2$ m, $a = 1.8$ m, 5.3 T), 1 MeV deuterium tangent to $R = 5.3$ m, along and against the field, 300 markers with two births each.

In [ ]:
iter_eq = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
iter_plasma = foc.Plasma.parabolic(
    1e20, 2e4, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)}
)
iter_edges = np.linspace(0.0, 1.0, 21)
iter_surfaces = iter_eq.flux_surfaces(iter_edges)
iter_field = orbits.orbit_field(iter_eq, (128, 4, 128))
iter_runs = {}
for sense, label in ((1, "co"), (-1, "counter")):
    origin, direction = foc.tangential_injection(5.3, 12.0, sense=sense)
    beam = foc.NeutralBeam.gaussian(
        origin,
        direction,
        1.0e6,
        16.5e6,
        width=0.3,
        divergence=0.005,
        markers=300,
        seed=1,
    )
    births_iter = foc.deposit(
        beam,
        iter_eq,
        iter_plasma,
        step=0.03,
        births_per_marker=2,
        rho_edges=iter_edges,
        seed=2,
    )
    start = time.perf_counter()
    averaged = orbit_average_births(
        births_iter,
        iter_eq,
        field=iter_field,
        rho_limit=np.sqrt(1.3),
        step=0.2,
        interval=1.0,
    )
    zero = foc.slow_down(births_iter, iter_plasma, iter_surfaces)
    orbit = foc.slow_down(averaged, iter_plasma, iter_surfaces)
    iter_runs[label] = (averaged, zero, orbit)
    w = np.asarray(births_iter.births.weights)
    ok = ~averaged.prompt_lost
    mean_shift = (
        np.sum(w[ok] * (averaged.mean_rho[ok] - np.asarray(births_iter.birth_rho)[ok]))
        / w[ok].sum()
    )
    print(
        f"{label:8s} {time.perf_counter() - start:4.1f} s: prompt losses {averaged.prompt_loss_fraction:.2%}, "
        f"mean shift in rho {mean_shift:+.3f}, power kept on the birth shell "
        f"{np.sum(np.diag(averaged.redistribution()) * averaged.to_dataset().zero_width_power) / averaged.confined_power:.0%}"
    )
    print(
        f"         driven current {zero.driven_current / 1e6:+.3f} MA at zero width, {orbit.driven_current / 1e6:+.3f} MA first orbit"
    )

changes = []
for averaged, _, _ in iter_runs.values():
    data = averaged.to_dataset(iter_surfaces)
    inside = np.cumsum(data.power.values - data.zero_width_power.values) / 1e6
    changes.append(
        xr.DataArray(
            inside, dims="rho", coords={"rho": ("rho", iter_edges[1:], data.rho.attrs)}
        )
    )
models(
    changes, list(iter_runs), "power inside rho, first orbit minus birth shells", "MW"
).plasma.plot.profiles(x="rho", over="model", title="deposited power inside rho")

A 1 MeV deuteron in 5.3 T has a Larmor radius of about 4 cm, against shells 9 cm wide: about a third of the power leaves its birth shell, mostly for a neighbour. On average the births along the field move outward by 0.014 in $\rho$ and those against it inward by as much, the direction the conservation of $p_\varphi$ gives for births on the outboard side (the tests check the sign and the size of the shift for single orbits). The power inside each $\rho$ shows it: less of it inside for the beam along the field, more for the one against it, by up to 0.6 to 0.7 MW of the 16.5 MW near $\rho = 0.8$. No birth is lost promptly.

In [ ]:
averaged, zero, orbit = iter_runs["co"]
current = models(
    [
        zero.to_dataset(iter_surfaces).current_density,
        orbit.to_dataset(iter_surfaces).current_density,
    ],
    ["zero width", "first orbit"],
    "driven current density",
    "A/m^2",
)
current.plasma.plot.profiles(x="rho", over="model")

A 1 MeV deuteron in 5.3 T has a Larmor radius of about 4 cm, against shells 9 cm wide: about a third of the power leaves its birth shell, mostly for a neighbour. On average the births along the field move outward by 0.014 in $\rho$ and those against it inward by as much, the direction the conservation of $p_\varphi$ gives for births on the outboard side (the tests check the sign and the size of the shift for single orbits). The power inside each $\rho$ shows it: less of it inside for the beam along the field, more for the one against it, by up to 0.6 to 0.7 MW of the 16.5 MW near $\rho = 0.8$. No birth is lost promptly._CURRENT

## Prompt losses against the field and the plasma current

In the small tokamak the orbit widths grow as $1/B$ at a fixed safety factor, and as $q \propto 1/I_p$ at a fixed field. The Solov'ev flux scales as $B_0 / q_0$, so the plasma current does too; it is computed here from Ampère's law around the last closed surface, $I_p = \oint \mathbf{B}_p \cdot d\mathbf{l} / \mu_0$. The same beam (600 markers) at three fields and four safety factors on the axis, with prompt losses at the wall:

In [ ]:
def plasma_current(equilibrium):
    """I_p from the poloidal field around the last closed surface [A]."""
    v = np.asarray(equilibrium.boundary.vertices, dtype=float)
    mid = 0.5 * (v[1:] + v[:-1])
    b_r, b_z, _ = equilibrium.magnetic_field(mid[:, 0], mid[:, 1])
    dl = np.diff(v, axis=0)
    return (
        abs(float(np.sum(np.asarray(b_r) * dl[:, 0] + np.asarray(b_z) * dl[:, 1])))
        / mu_0
    )


fields, safety = [1.0, 1.5, 2.5], [1.0, 2.0, 3.0, 4.0]
loss = np.zeros((len(fields), len(safety)))
current_ma = np.zeros_like(loss)
for i, b0 in enumerate(fields):
    scan = ascot5.benchmark_case(markers=600, toroidal_field=b0)
    for j, q0 in enumerate(safety):
        eq = foc.Equilibrium.solovev(1.65, 0.5, 1.6, b0, q0, wall_psi_n=1.3)
        births_scan = foc.deposit(
            scan.beam,
            eq,
            scan.plasma,
            step=0.005,
            births_per_marker=1,
            rho_edges=rho_edges,
            seed=2,
        )
        averaged = orbit_average_births(
            births_scan, eq, rho_limit=wall_rho, interval=0.5
        )
        loss[i, j] = 100 * averaged.prompt_loss_fraction
        current_ma[i, j] = plasma_current(eq) / 1e6
print("B [T]  " + "".join(f"   q0 = {q0:.0f}: I_p, loss" for q0 in safety))
for i, b0 in enumerate(fields):
    print(
        f"{b0:5.1f}  "
        + "".join(
            f"{current_ma[i, j]:12.2f} MA {loss[i, j]:5.2f}%"
            for j in range(len(safety))
        )
    )
losses = xr.DataArray(
    loss,
    dims=("B", "q0"),
    coords={
        "B": ("B", fields, {"long_name": "$B_0$", "units": "T"}),
        "q0": ("q0", safety, {"long_name": "$q$ on the axis"}),
    },
    name="prompt losses",
    attrs={"long_name": "prompt losses", "units": "%"},
)
losses.plasma.plot.profiles(x="q0", over="B")

At $q_0 = 1$ hardly any birth is lost; the losses grow as the current falls, to 12% at 2.5 T and 1.2 MA and 71% at 1 T and 0.47 MA. They follow the plasma current rather than the field: at 0.94 MA they are 19% at 1 T and 22% at 1.5 T, at 1.4 to 1.6 MA 4% at 1.5 T and 3% at 2.5 T. A banana's width scales with the poloidal Larmor radius $m v / (e B_p)$, and $B_p$ with $I_p$. (The currents are those of this Solov'ev shape, whose $q$ is low at the edge; the trend, not the values, carries over.)

## Zero width, first orbit and orbit following

How close does the first orbit come to following the orbits through the whole slowing down? `follow_orbits` on the same 200 births of the 2.5 T case (Monte Carlo collisions without energy diffusion, which `slow_down` leaves out; the same end energy, $\max(2T_i, 1\,\mathrm{keV})$; losses at the wall), against `slow_down` of the births at zero width and averaged over their first orbit. The gap is the sum over the shells of the difference of the normalised profiles (0: the same shape, 2: no overlap).

In [ ]:
end = {"thermal_factor": 2.0, "minimum_energy": 1e3}
zero = foc.slow_down(deposition, plasma, surfaces, **end)
orbit = foc.slow_down(first, plasma, surfaces, **end)
start = time.perf_counter()
followed = orbits.follow_orbits(
    deposition,
    field,
    plasma,
    surfaces=surfaces,
    rho_limit=wall_rho,
    step=0.25,
    energy_diffusion=False,
    seed=3,
    **end,
)
print(
    f"follow_orbits: {time.perf_counter() - start:.0f} s, fates { ({k: round(v, 3) for k, v in followed.fate_fractions().items() if v}) }"
)
followed_data = followed.to_dataset()


def gap(a, b):
    return float(np.sum(np.abs(a / a.sum() - b / b.sum())))


per_shell = {
    "electron heating [W]": (
        zero.electron_heating,
        orbit.electron_heating,
        followed.electron_heating,
    ),
    "heating [W]": (
        zero.electron_heating + zero.ion_heating,
        orbit.electron_heating + orbit.ion_heating,
        followed.electron_heating + followed.ion_heating,
    ),
    "fast ions": (
        zero.fast_ion_density * surfaces.volume,
        orbit.fast_ion_density * surfaces.volume,
        followed.particles,
    ),
    "fast-ion current [A]": (
        zero.fast_ion_current_density * surfaces.area,
        orbit.fast_ion_current_density * surfaces.area,
        followed_data.fast_ion_current_density.values * surfaces.area,
    ),
}
print(
    f"{'':22s}{'zero width':>12s}{'first orbit':>12s}{'orbits':>12s}   gap: zero width, first orbit"
)
for name, (a, b, c) in per_shell.items():
    print(
        f"{name:22s}{a.sum():12.4g}{b.sum():12.4g}{c.sum():12.4g}   {gap(a, c):.3f}, {gap(b, c):.3f}"
    )
density = models(
    [
        zero.to_dataset(surfaces).fast_ion_density,
        orbit.to_dataset(surfaces).fast_ion_density,
        followed_data.fast_ion_density,
    ],
    ["zero width", "first orbit", "orbit following"],
    "fast-ion density",
    "1/m^3",
)
density.plasma.plot.profiles(x="rho", over="model").show()
current = models(
    [
        zero.to_dataset(surfaces).fast_ion_current_density,
        orbit.to_dataset(surfaces).fast_ion_current_density,
        followed_data.fast_ion_current_density,
    ],
    ["zero width", "first orbit", "orbit following"],
    "fast-ion current density",
    "A/m^2",
)
current.plasma.plot.profiles(x="rho", over="model")

For the heating and the fast-ion content the first orbit takes most of the way from zero width to orbit following: the gap of the heating profile falls from 0.26 to 0.08, that of the fast-ion density from 0.26 to 0.08, and the electron heating rises from 1.107 to 1.169 MW against 1.174 MW with orbits. The current is the exception: its gap falls only from 0.26 to 0.24, and the totals move apart, 3% below zero width with the first orbit (65.6 against 67.6 kA) and 5% above it with orbit following (71.0 kA). The first orbit moves the births but not what happens to them while they slow down; pitch-angle scattering moving ions between trapped and passing orbits on the way, which a local model does not see, is a likely part of the rest. With 200 markers the Monte Carlo noise of the collisions is several percent per shell.

## The ASCOT5 benchmark's current

The full benchmark case (2000 markers, four births each, as `ascot5.run_freeofcharge`) with the first-orbit correction, against the stored ASCOT5 results (`benchmarks/ascot5/ascot5_results.json`, the run with orbits; ASCOT5 is not run here):

In [ ]:
full = ascot5.benchmark_case()
full_births = foc.deposit(
    full.beam,
    full.equilibrium,
    full.plasma,
    step=0.005,
    births_per_marker=4,
    rho_edges=full.rho_edges,
    seed=2,
)
start = time.perf_counter()
full_first = orbit_average_births(
    full_births, full.equilibrium, field=field, rho_limit=wall_rho
)
print(f"{len(full_first.orbit_time)} births in {time.perf_counter() - start:.1f} s")
reference = ascot5.load_results(Path("../benchmarks/ascot5/ascot5_results.json"))
end = {"thermal_factor": 2.0, "minimum_energy": 1e3}
results = {
    "zero width": foc.slow_down(full_births, full.plasma, surfaces, **end),
    "first orbit": foc.slow_down(full_first, full.plasma, surfaces, **end),
}
print(
    f"{'':12s}{'fast-ion current':>18s}{'electron heating':>18s}{'fast ions':>12s}{'trapped':>9s}"
)
for name, steady in results.items():
    i_fast = float(np.sum(steady.fast_ion_current_density * surfaces.area))
    print(
        f"{name:12s}{i_fast / 1e3:15.1f} kA{steady.electron_heating.sum() / 1e6:15.3f} MW"
        f"{np.sum(steady.fast_ion_density * surfaces.volume):12.3g}{steady.trapped_births:9.0%}"
    )
print(
    f"{'ASCOT5':12s}{reference['fast_ion_current'] / 1e3:15.1f} kA"
    f"{sum(reference['electron_heating']) / 1e6:15.3f} MW{reference['fast_ions']:12.3g}"
)
print(
    f"prompt losses {full_first.prompt_loss_fraction:.2%} of the power (ASCOT5: {reference['lost_power']:.0f} W lost)"
)

The first-orbit correction lowers the benchmark's fast-ion current by 3% (69.4 to 67.5 kA), not the 18% to ASCOT5's 58.8 kA. It does move 5% of the heating to the electrons (1.121 to 1.178 MW), the shift ASCOT5's orbits give (1.171 MW without them, 1.227 MW with them). No birth is lost at the wall, as in ASCOT5. With freeofcharge's own orbit following the current of the births above rises instead of falling, so in freeofcharge's guiding-centre model the orbit width does not explain the current difference to ASCOT5; the question of the benchmark page stays open, and the field scan in ASCOT5 with enough CPU time per marker remains its test.